# 원판 검출 평가 전 수동 판독
기존 자료의 A그룹 2명과 C그룹 1명, 7편. 먼저 원본만 보고 동작 시작과 끝을 기록한다. 구간의 양 끝과 25%, 50%, 75% 지점에 가까운 프레임에서 가까운 원판 한 개를 확인한다. 모델은 학습하거나 실행하지 않는다. VS Code에서 프로젝트의 `.venv` 커널을 선택한다.

최초 14명 및 이후 원판 모델·추적 규칙 비교 기록과 대조해 정한 후향 평가 후보다. 원판 없는 영상을 찾으면서 원본의 표본 장면을 본 이력은 있다. 기록 밖에서 후보 영상의 예측을 보고 설정을 바꾼 이력이 있다면 채점 전에 적어 둔다.

전체 분할표에서 세 선수는 `test`로 배정되어 있다. 판독은 프로젝트의 `data/annotations/plate_holdout/` 아래에 저장한다.

## 1. 영상 선택
드롭다운에서 영상을 바꾼 뒤 2번과 3번 편집기 셀을 다시 실행한다. 저장된 수동 기록은 다시 열 수 있다.

In [ ]:
import hashlib

import ipywidgets as widgets
import pandas as pd
from IPython.display import display
from wlcv.config import ANNOTATIONS, SPLIT, VIDEOS
from wlcv.annotation.events import phase_event_editor
from wlcv.annotation.boxes import plate_box_editor
from wlcv.annotation.storage import box_path, load_events
from wlcv.data.dataset import read_splits

videos = VIDEOS
review = ANNOTATIONS / "plate_holdout"
roles = read_splits(SPLIT)
test_ids = {video_id for video_id, role in roles.items() if role == "test"}
split = pd.read_csv(SPLIT).set_index("video_id")
cohort = pd.read_csv(SPLIT.with_name("plate_holdout_videos.csv"))
if cohort.video_id.duplicated().any() or set(cohort.video_id) != test_ids:
    raise ValueError("전체 분할표의 test 영상과 고정 목록이 다릅니다.")
for item in cohort.itertuples():
    if split.at[item.video_id, "person_id"] != item.person_id:
        raise ValueError(f"{item.video_id}의 선수 ID가 고정 목록과 다릅니다.")
    source = videos / (item.video_id + ".mp4")
    if hashlib.sha256(source.read_bytes()).hexdigest() != item.sha256:
        raise ValueError(f"{source.name}의 원본 파일이 고정 목록과 다릅니다.")

choose = widgets.Dropdown(options=cohort.video_id.tolist(), description="영상")
display(choose)

## 2. 동작 시작과 끝 저장
프레임 번호는 0부터 시작한다. 원본에서 `ready`와 `finish`를 각각 찾은 뒤 **현재 프레임 기록**을 누른다. 이름에 `_h_`가 있는 하프 영상은 끝에 `drive`를 기록한다. 끝을 판단하기 어려우면 비워 두고 사유를 남긴다. 구간을 저장한 뒤 3번 셀을 실행한다.

In [ ]:
video = videos / f"{choose.value}.mp4"
display(phase_event_editor(video, pass_number=1, annotations=review))

## 3. 고정한 다섯 장에서 박스 그리기
목록의 프레임을 선택하고 카메라 쪽에서 **실제로 들어 올리는 원판**의 보이는 외곽을 한 번 드래그한 뒤 **박스 저장**을 누른다. **원판 없음 저장**은 목표 원판이 없다고 확인한 경우에만 사용한다. 가림이나 잘림 때문에 정답을 그릴 수 없으면 해당 프레임에서 **판독 불가 기록**을 누른다. 이 기록은 채점에서 우선하며, 다시 판독할 수 있으면 해제한다. 영상을 바꾸기 전에 현재 작업을 저장한다.

In [ ]:
def sampled_frames(video_id):
    events = load_events(review, video_id, 1)
    last_event = "drive" if "_h_" in video_id else "finish"
    if "ready" not in events or last_event not in events:
        return []
    start, end = events["ready"], events[last_event]
    if end <= start:
        raise ValueError(f"{video_id}의 시작과 끝 동작 번호를 확인하세요.")
    return sorted({start + round((end-start)*i/4) for i in range(5)})

def uncertain_file(video_id, frame):
    return review / "판독불가" / f"{video_id}_f{frame}.txt"

def open_boxes(video_id):
    targets = sampled_frames(video_id)
    print(video_id, "대상 프레임", targets)
    if not targets:
        print("먼저 2번에서 동작 시작과 끝을 저장하세요.")
        return
    editor = plate_box_editor(videos / f"{video_id}.mp4", annotations=review)
    slider = next(item for item in editor.children if isinstance(item, widgets.IntSlider))
    slider.value = targets[0]
    frame_select = widgets.Dropdown(options=targets, description="대상 프레임")
    message = widgets.Label()
    def jump(change):
        if slider.disabled:
            print("편집 중인 박스를 저장하거나 다시 읽은 뒤 이동하세요.")
            return
        slider.value = change["new"]
    frame_select.observe(jump, names="value")
    mark = widgets.Button(description="판독 불가 기록")
    clear = widgets.Button(description="판독 불가 해제")
    def update_flag(button):
        frame = slider.value
        if frame not in targets:
            message.value = "대상 프레임에서 기록하세요."
            return
        path = uncertain_file(video_id, frame)
        if button is mark:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text("판독 불가", encoding="utf-8")
        else:
            path.unlink(missing_ok=True)
        message.value = f"{video_id} {frame}번 · {button.description}"
    mark.on_click(update_flag)
    clear.on_click(update_flag)
    display(frame_select, editor, widgets.HBox([mark, clear]), message)

open_boxes(choose.value)

## 4. 저장 현황 확인
라벨을 저장한 뒤 이 셀을 다시 실행한다. 구간 미기록, 박스 미기록, 확인한 빈 장면, 판독 불가를 구분한다. 우선 이 7편에서 판독 가능한 사진의 동작 중 미탐·오탐을 집계할 예정이다. 프레임 표본 사이에서 연속으로 놓친 길이는 이 표만으로 채점할 수 없다.

In [ ]:
from collections import Counter
progress = []
for video_id in cohort.video_id:
    targets = sampled_frames(video_id)
    counts = Counter()
    for frame in targets:
        box = box_path(review, video_id, frame)
        if uncertain_file(video_id, frame).exists():
            state = "판독 불가"
        elif not box.exists():
            state = "미기록"
        else:
            state = "원판 있음" if box.read_text().strip() else "원판 없음"
        counts[state] += 1
    progress.append([video_id, "완료" if targets else "구간 미기록",
                     len(targets), counts["원판 있음"], counts["원판 없음"],
                     counts["판독 불가"], counts["미기록"]])
display(pd.DataFrame(progress, columns=["영상","구간","대상","원판 있음","원판 없음","판독 불가","박스 미기록"]))